# THOP으로 간단하게 MMAC 측정하기

이 노트북은 `THOP` 패키지로 모델 1회 MAC을 측정하고, 실제 추론에서 호출하는 모든 `(모델, 입력 view)`의 MAC을 `sum()`으로 더합니다.

현재 CNN처럼 입력값에 따라 실행 경로가 바뀌지 않는 모델은 실제 데이터가 필요하지 않습니다. **실제와 같은 입력 shape**의 zero tensor면 충분합니다.

In [ ]:
!pip install thop

In [ ]:
from collections.abc import Sequence

import pandas as pd
import torch
from torch import nn
from thop import profile
from thop.vision.basic_hooks import zero_ops

LIMIT_MMAC = 100.0

## 베이스라인 모델 구조

아래 셀에 베이스라인의 연산 구조를 직접 포함했습니다. 별도의 `src` 파일이나 checkpoint 없이 이 노트북만 실행할 수 있습니다. 가중치 값은 MAC에 영향을 주지 않으므로 임의로 초기화된 모델을 사용합니다.

In [ ]:
INPUT_FEATURE_SHAPE = (5, 64, 201)
DEFAULT_CHANNELS = (32, 48, 48, 64, 64, 96, 96, 128, 160)
DEFAULT_STRIDES = (
    (1, 1), (1, 1), (2, 2), (1, 1),
    (2, 2), (1, 1), (2, 2), (1, 1),
)


class ConvNormActivation(nn.Sequential):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: tuple[int, int] = (1, 1),
        groups: int = 1,
    ) -> None:
        super().__init__(
            nn.Conv2d(
                in_channels, out_channels, kernel_size,
                stride=stride, padding=kernel_size // 2,
                groups=groups, bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )


class DepthwiseSeparableBlock(nn.Module):
    def __init__(
        self, in_channels: int, out_channels: int, stride: tuple[int, int]
    ) -> None:
        super().__init__()
        self.depthwise = ConvNormActivation(
            in_channels, in_channels, kernel_size=3,
            stride=stride, groups=in_channels,
        )
        self.pointwise = ConvNormActivation(
            in_channels, out_channels, kernel_size=1
        )
        self.use_residual = stride == (1, 1) and in_channels == out_channels

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        outputs = self.pointwise(self.depthwise(inputs))
        return outputs + inputs if self.use_residual else outputs


class SpatialBaselineCNN(nn.Module):
    def __init__(
        self,
        input_channels: int = 5,
        num_classes: int = 8,
        num_azimuths: int = 25,
        channels: Sequence[int] = DEFAULT_CHANNELS,
        dropout: float = 0.15,
    ) -> None:
        super().__init__()
        channels = tuple(int(channel) for channel in channels)
        if len(channels) != len(DEFAULT_STRIDES) + 1:
            raise ValueError("channels 길이가 올바르지 않습니다")

        self.stem = ConvNormActivation(
            input_channels, channels[0], kernel_size=3, stride=(2, 2)
        )
        self.blocks = nn.Sequential(*[
            DepthwiseSeparableBlock(in_ch, out_ch, stride)
            for in_ch, out_ch, stride in zip(
                channels[:-1], channels[1:], DEFAULT_STRIDES
            )
        ])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.sound_head = nn.Linear(channels[-1], num_classes)
        self.azimuth_head = nn.Linear(channels[-1], num_azimuths)

    def forward(self, inputs: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        embedding = self.pool(self.blocks(self.stem(inputs))).flatten(1)
        embedding = self.dropout(embedding)
        return self.sound_head(embedding), self.azimuth_head(embedding)


def mirror_spatial_features(features: torch.Tensor) -> torch.Tensor:
    """[L, R, ILD, cos(IPD), sin(IPD)] 특징을 좌우 반전합니다."""
    mirrored = features[:, (1, 0, 2, 3, 4)].clone()
    mirrored[:, 2].neg_()
    mirrored[:, 4].neg_()
    return mirrored

## 1. 모델 1회 측정

THOP의 `profile()`이 핵심입니다. 프로젝트의 기존 31.372064 MMAC 기준과 맞추기 위해 BN, activation, pooling, dropout은 0-op으로 지정합니다.

In [ ]:
PROJECT_OP_RULES = {
    nn.BatchNorm2d: zero_ops,
    nn.SiLU: zero_ops,
    nn.AdaptiveAvgPool2d: zero_ops,
    nn.Dropout: zero_ops,
}


def mmac(model: nn.Module, sample: torch.Tensor) -> float:
    """한 번의 forward에 대한 MMAC을 반환합니다."""
    if sample.shape[0] != 1:
        raise ValueError("샘플당 MMAC은 batch size 1로 측정하세요")
    model.eval()
    with torch.inference_mode():
        macs, _ = profile(
            model,
            inputs=(sample,),
            custom_ops=PROJECT_OP_RULES,
            verbose=False,
        )
    return float(macs) / 1_000_000.0


model = SpatialBaselineCNN().eval()
dummy_input = torch.zeros(1, *INPUT_FEATURE_SHAPE)
single_mmac = mmac(model, dummy_input)

print(f"input: {tuple(dummy_input.shape)}")
print(f"parameters: {sum(parameter.numel() for parameter in model.parameters()):,}")
print(f"single forward: {single_mmac:.6f} MMAC")

## 2. 단일 모델, TTA, 앙상블을 같은 방식으로 계산

각 시나리오를 실제 실행되는 `(model, input)` 목록으로 표현합니다. TTA는 같은 모델이 여러 번 등장하고, 앙상블은 서로 다른 모델 객체가 등장합니다. 앙상블과 TTA를 함께 쓰면 두 경우가 모두 들어갑니다.

In [ ]:
def total_mmac(calls: list[tuple[nn.Module, torch.Tensor]]) -> float:
    return sum(mmac(current_model, current_input) for current_model, current_input in calls)


model_a = model
model_b = SpatialBaselineCNN().eval()  # 두 번째 checkpoint라고 가정
tta_views = [dummy_input, mirror_spatial_features(dummy_input)]
ensemble_models = [model_a, model_b]

scenarios = {
    "single model": [(model_a, dummy_input)],
    "single model + 2-view TTA": [(model_a, view) for view in tta_views],
    "2-model ensemble": [(current_model, dummy_input) for current_model in ensemble_models],
    "2-model ensemble + 2-view TTA": [
        (current_model, view)
        for current_model in ensemble_models
        for view in tta_views
    ],
}

rows = []
for name, calls in scenarios.items():
    current_total = total_mmac(calls)
    rows.append({
        "scenario": name,
        "forward_calls": len(calls),
        "total_mmac": current_total,
        "under_100_mmac": current_total < LIMIT_MMAC,
    })

result = pd.DataFrame(rows)
result

기본 모델의 예상 결과는 다음과 같습니다.

| 시나리오 | forward 횟수 | 총 MMAC |
|---|---:|---:|
| 단일 모델 | 1 | 31.372064 |
| 단일 모델 + TTA | 2 | 62.744128 |
| 2-model 앙상블 | 2 | 62.744128 |
| 2-model 앙상블 + TTA | 4 | 125.488256 |

즉, TTA를 batch로 묶어 동시에 실행하더라도 총 MAC은 모든 view를 합산해야 합니다.

## 3. 입력 크기나 모델 구조가 서로 다른 경우

이 경우에도 실제 `(model, input)` 쌍을 목록에 넣기만 하면 됩니다. 예를 들어 full view와 짧은 crop은 각각 프로파일링한 뒤 자동으로 합산됩니다.

In [ ]:
short_crop = dummy_input[..., :161]
different_shape_calls = [(model_a, dummy_input), (model_a, short_crop)]
print(f"full + short-crop TTA: {total_mmac(different_shape_calls):.6f} MMAC")

## 내 모델에 적용하기

`model_a`, `model_b`, `dummy_input`을 실제 모델과 입력 shape으로 바꾸고 호출 목록만 작성하면 됩니다.

```python
calls = [
    (model_a, input_a),       # model A 원본
    (model_a, tta_input_a),   # model A TTA
    (model_b, input_b),       # model B 원본
]
print(f"total: {total_mmac(calls):.6f} MMAC")
```

주의할 점은 세 가지입니다.

- 반드시 batch size 1과 실제 입력 shape을 사용합니다.
- early-exit나 조건부 routing 모델은 입력값에 따라 연산량이 달라지므로 대표 입력과 최악 입력을 따로 측정합니다.
- Conv1d, RNN, attention, functional 연산, custom op를 쓰는 새 구조는 THOP 지원 여부를 확인하고 필요한 `custom_ops`를 등록해야 합니다. 이 노트북의 `PROJECT_OP_RULES`는 현재 baseline의 공식 수치와 맞추기 위한 설정입니다. 순수 THOP 기본값을 원하면 `custom_ops=PROJECT_OP_RULES` 인자를 제거하면 됩니다.